# Thai Character & Number Recognition — ResNet50 (เฉพาะ D2)

Fine-tune **ResNet50 (ImageNet-pretrained)** โดยใช้ข้อมูลจากแหล่ง **D2 (THI - C168)** อย่างเดียว

- ข้อมูล: `Dataset/character/<code>/D2/*` และ `Dataset/number/<code>/D2/*` รวม ราว 76,900 รูป
- แต่ละโฟลเดอร์ `<code>` คือ 1 คลาส (รหัส TIS-620 เช่น 161 = ก, 240 = ๐)
- แบ่งข้อมูลเป็น **train 80% / validation 20%** (ไม่มี test set แยก — วัดผลบน validation)
- D2 ไม่มีรูปของเลขไทยทั้งหมดและสระบางตัว แต่คลาสที่มีนั้นมีรูปเยอะและสมดุลมาก (คลาสละ ~990–1,195 รูป)
- ใช้ PyTorch + GPU ของ Mac (MPS) — TensorFlow ในเครื่องรันได้แค่ CPU

## 1. Setup & Config

In [ ]:
import json, random, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.model_selection import train_test_split
from torchvision import models

# หา root ของโปรเจกต์ (รันได้ทั้งจาก root หรือจากโฟลเดอร์ Dataset/)
ROOT = Path.cwd()
while not (ROOT / "Dataset" / "character").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

CONFIG = dict(
    source="D2",          # ใช้เฉพาะแหล่งนี้ ("D1", "D2", "D3" หรือ None = ใช้ทั้งหมด)
    val_ratio=0.2,        # train 80% / validation 20%
    min_per_class=5,      # คลาสที่มีรูปน้อยกว่านี้จะถูกตัดออก
    img_size=96,
    batch_size=128,
    epochs=15,
    lr=1e-3,
    seed=42,
    subset=None,          # เช่น 0.1 = ใช้แค่ 10% ของข้อมูลเพื่อทดลองรันเร็ว ๆ, None = ใช้ทั้งหมด
    out_dir="outputs/resnet50_d2",
)

random.seed(CONFIG["seed"]); np.random.seed(CONFIG["seed"]); torch.manual_seed(CONFIG["seed"])
device = torch.device("mps" if torch.backends.mps.is_available()
                      else "cuda" if torch.cuda.is_available() else "cpu")
OUT = ROOT / CONFIG["out_dir"]
OUT.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT, "| device:", device)

## 2. โหลดและ Preprocess รูป

รูปของ D2 เป็นภาพสี ขนาดราว 145×140 px และมีขอบขาวรอบตัวอักษรเยอะ จึงทำให้เป็นมาตรฐานเดียวกัน:
**grayscale → crop เฉพาะส่วนที่เป็นหมึก → pad เป็นสี่เหลี่ยมจัตุรัส → resize 96×96**

Cache `data/cache_96.npz` ใช้ร่วมกับโน้ตบุ๊กหลัก (เก็บรูปของทุกแหล่ง) แล้วค่อยกรองเฉพาะ D2 ในขั้นตอนถัดไป
ถ้ายังไม่เคยสร้าง cache รอบแรกจะใช้เวลาหลายนาที

In [ ]:
DATA_DIRS = [ROOT / "Dataset" / "character", ROOT / "Dataset" / "number"]
EXTS = {".jpg", ".jpeg", ".png", ".bmp"}


def preprocess(path, size):
    """Grayscale -> crop to ink bounding box -> pad to square -> resize. Returns uint8 HxW."""
    arr = np.asarray(Image.open(path).convert("L"))
    ink = arr < 128  # ทุกแหล่งเป็นหมึกเข้มบนพื้นขาว
    if ink.any():
        ys, xs = np.where(ink)
        arr = arr[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    h, w = arr.shape
    side = int(max(h, w) * 1.15) + 2
    canvas = np.full((side, side), 255, np.uint8)
    y0, x0 = (side - h) // 2, (side - w) // 2
    canvas[y0:y0 + h, x0:x0 + w] = arr
    return np.asarray(Image.fromarray(canvas).resize((size, size), Image.BILINEAR))


def scan():
    samples = []
    for d in DATA_DIRS:
        for code_dir in sorted(d.iterdir()):
            if not code_dir.is_dir() or code_dir.name.startswith("_"):
                continue  # ข้ามโฟลเดอร์พิเศษ เช่น _review ที่ clean_d1.py สร้างไว้
            for f in code_dir.rglob("*"):
                if f.suffix.lower() in EXTS:
                    source = f.parent.name if f.parent != code_dir else "?"
                    samples.append((str(f), code_dir.name, source))
    return samples


def build_cache(size, cache_path):
    if cache_path.exists():
        z = np.load(cache_path, allow_pickle=True)
        return z["x"], z["codes"], z["sources"]
    samples = scan()
    print(f"Preprocessing {len(samples)} images to {size}x{size} (one-time cache)...")
    x = np.empty((len(samples), size, size), np.uint8)
    for i, (p, _, _) in enumerate(samples):
        x[i] = preprocess(p, size)
        if i % 20000 == 0:
            print(f"  {i}/{len(samples)}")
    codes = np.array([s[1] for s in samples])
    sources = np.array([s[2] for s in samples])
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez(cache_path, x=x, codes=codes, sources=sources)
    return x, codes, sources


x_raw, codes_raw, sources_raw = build_cache(CONFIG["img_size"], ROOT / "data" / f"cache_{CONFIG['img_size']}.npz")
codes_raw = np.array([str(c) for c in codes_raw])
sources_raw = np.array([str(s) for s in sources_raw])
print(f"cache: {len(codes_raw)} images from all sources")

### กรองเฉพาะ D2 และตัดคลาสที่มีรูปน้อยเกินไป

คลาสที่มีรูปเพียง 1–4 รูปจะแบ่ง train/validation ไม่ได้ และเรียนรู้อะไรไม่ได้จริง จึงตัดออกและพิมพ์รายชื่อไว้ให้ตรวจสอบ

In [ ]:
import collections

mask = np.ones(len(codes_raw), bool) if CONFIG["source"] is None else (sources_raw == CONFIG["source"])
counts_all = collections.Counter(codes_raw[mask].tolist())
missing = sorted(set(codes_raw.tolist()) - set(counts_all), key=int)
too_few = sorted([c for c, n in counts_all.items() if n < CONFIG["min_per_class"]], key=int)

thai = lambda code: bytes([int(code)]).decode("tis-620")  # 161 -> ก
print(f"คลาสที่ {CONFIG['source']} ไม่มีรูปเลย ({len(missing)}): " +
      ", ".join(f"{c}({thai(c)})" for c in missing))
print(f"คลาสที่มีรูปน้อยกว่า {CONFIG['min_per_class']} รูป จึงถูกตัดออก ({len(too_few)}): " +
      ", ".join(f"{c}({thai(c)})×{counts_all[c]}" for c in too_few))

keep = {c for c, n in counts_all.items() if n >= CONFIG["min_per_class"]}
mask &= np.isin(codes_raw, list(keep))

x_all, codes, sources = x_raw[mask], codes_raw[mask], sources_raw[mask]
classes = sorted({str(c) for c in codes}, key=int)  # plain str (torch.load safe)
cls_idx = {c: i for i, c in enumerate(classes)}
y_all = np.array([cls_idx[str(c)] for c in codes], np.int64)
print(f"\nใช้งานจริง: {len(y_all)} images, {len(classes)} classes")
print("".join(thai(c) for c in classes))

### ดูข้อมูล: จำนวนรูปต่อคลาส และตัวอย่างรูปหลัง preprocess

In [ ]:
counts = np.bincount(y_all, minlength=len(classes))
plt.figure(figsize=(16, 3))
plt.bar(range(len(classes)), counts)
plt.xticks(range(len(classes)), classes, rotation=90, fontsize=7)
plt.title(f"Images per class — {CONFIG['source']}"); plt.tight_layout(); plt.show()
print(f"น้อยสุด {counts.min()} | มากสุด {counts.max()} | เฉลี่ย {counts.mean():.0f} รูปต่อคลาส")

rng = np.random.default_rng(0)
ids = rng.choice(len(y_all), 24, replace=False)
fig, axes = plt.subplots(2, 12, figsize=(14, 3))
for ax, i in zip(axes.flat, ids):
    ax.imshow(x_all[i], cmap="gray"); ax.set_title(codes[i], fontsize=8); ax.axis("off")
plt.suptitle(f"Samples after preprocessing — {CONFIG['source']}"); plt.tight_layout(); plt.show()

## 3. แบ่งข้อมูล train 80% / validation 20% (stratified ตามคลาส)

In [ ]:
idx = np.arange(len(y_all))
if CONFIG["subset"]:
    idx, _ = train_test_split(idx, train_size=CONFIG["subset"], stratify=y_all, random_state=CONFIG["seed"])
tr, va = train_test_split(idx, test_size=CONFIG["val_ratio"], stratify=y_all[idx], random_state=CONFIG["seed"])
np.savez(OUT / "split.npz", train=tr, val=va)
print(f"train {len(tr)} | val {len(va)}")

X = torch.from_numpy(x_all)   # uint8 (N, H, W) อยู่บน CPU, ส่งขึ้น GPU ทีละ batch
Y = torch.from_numpy(y_all)

## 4. Data augmentation (ทำบน GPU ทีละ batch)

สุ่ม rotate ±10°, shear ±8°, scale 0.85–1.1, translate ±8% ให้แต่ละรูป — ช่วยให้โมเดลไม่ยึดติดกับตำแหน่ง/ความเอียงของลายมือ

In [ ]:
def to_input(xb_uint8, train):
    """uint8 (B,H,W) -> float (B,3,H,W) normalized, with optional augmentation."""
    x = xb_uint8.to(device, non_blocking=True).float().div_(255).unsqueeze(1)
    if train:
        B = x.size(0)
        rnd = lambda lo, hi: torch.empty(B, device=device).uniform_(lo, hi)
        ang, shear = torch.deg2rad(rnd(-10, 10)), torch.deg2rad(rnd(-8, 8))
        scale = rnd(0.85, 1.1)
        cos, sin, tsh = torch.cos(ang), torch.sin(ang), torch.tan(shear)
        theta = torch.stack([
            torch.stack([cos / scale, (-sin + cos * tsh) / scale, rnd(-0.16, 0.16)], 1),
            torch.stack([sin / scale, (cos + sin * tsh) / scale, rnd(-0.16, 0.16)], 1),
        ], 1)
        grid = F.affine_grid(theta, x.shape, align_corners=False)
        # warp หมึก (1 - x) เพื่อให้ส่วนที่ล้นขอบถูกเติมเป็นพื้นขาว
        x = 1 - F.grid_sample(1 - x, grid, padding_mode="zeros", align_corners=False)
    x = (x - 0.5) / 0.5
    return x.expand(-1, 3, -1, -1)


def batches(ids, train):
    ids = torch.as_tensor(ids)
    if train:
        ids = ids[torch.randperm(len(ids))]
    bs = CONFIG["batch_size"]
    n = len(ids) // bs * bs if train else len(ids)  # drop_last ตอน train
    for s in range(0, n, bs):
        b = ids[s:s + bs]
        yield to_input(X[b], train), Y[b].to(device)


# ตัวอย่างรูปหลัง augment
xb, yb = next(batches(tr, train=True))
fig, axes = plt.subplots(2, 10, figsize=(13, 3))
for ax, img, lbl in zip(axes.flat, xb[:20, 0].cpu(), yb[:20].cpu()):
    ax.imshow(img, cmap="gray"); ax.set_title(classes[lbl], fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 5. สร้างโมเดล ResNet50 (pretrained ImageNet) แล้วเปลี่ยนชั้นสุดท้ายให้ตรงกับจำนวนคลาสของ D2

In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, len(classes))
model = model.to(device)

steps_per_epoch = len(tr) // CONFIG["batch_size"]
opt = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(
    opt, max_lr=CONFIG["lr"], total_steps=CONFIG["epochs"] * steps_per_epoch, pct_start=0.15)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M | steps/epoch: {steps_per_epoch}")

## 6. Train

บันทึกโมเดลที่ val accuracy ดีที่สุดไว้ที่ `outputs/resnet50_d2/best.pt` ทุก epoch — หยุดกลางทางได้ (Interrupt kernel) โดยไม่เสียโมเดลที่ดีที่สุด

D2 มีรูปราว 76,900 รูป (ประมาณ 47% ของข้อมูลทั้งหมด) 1 epoch ใช้เวลาราว 6–8 นาทีบน M-series Mac

In [ ]:
@torch.no_grad()
def evaluate(ids):
    model.eval()
    preds, loss_sum = [], 0.0
    for xb, yb in batches(ids, train=False):
        out = model(xb)
        loss_sum += F.cross_entropy(out, yb, reduction="sum").item()
        preds.append(out.argmax(1).cpu())
    preds = torch.cat(preds).numpy()
    return loss_sum / len(preds), preds


meta = {"classes": classes, "img_size": CONFIG["img_size"], "arch": "resnet50",
        "source": CONFIG["source"], "normalize": {"mean": 0.5, "std": 0.5},
        "preprocess": "gray, crop ink bbox, pad square x1.15"}
(OUT / "meta.json").write_text(json.dumps(meta, indent=2))

history, best_acc = [], 0.0
for ep in range(1, CONFIG["epochs"] + 1):
    model.train()
    t0, run_loss, correct, n = time.time(), 0.0, 0, 0
    for step, (xb, yb) in enumerate(batches(tr, train=True)):
        out = model(xb)
        loss = criterion(out, yb)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step(); sched.step()
        run_loss += loss.item() * len(yb); n += len(yb)
        correct += (out.argmax(1) == yb).sum().item()
        if step % 100 == 0:
            print(f"  ep{ep} step {step}/{steps_per_epoch} loss {loss.item():.4f}")
    val_loss, val_pred = evaluate(va)
    val_acc = float((val_pred == y_all[va]).mean())
    rec = dict(epoch=ep, train_loss=run_loss / n, train_acc=correct / n,
               val_loss=val_loss, val_acc=val_acc, sec=time.time() - t0)
    history.append(rec)
    print(f"Epoch {ep}/{CONFIG['epochs']} train_loss {rec['train_loss']:.4f} train_acc {rec['train_acc']:.4f} "
          f"val_loss {val_loss:.4f} val_acc {val_acc:.4f} ({rec['sec']:.0f}s)")
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({"model": model.state_dict(), **meta}, OUT / "best.pt")
    (OUT / "history.json").write_text(json.dumps(history, indent=2))
print("best val acc:", best_acc)

## 7. กราฟ Loss / Accuracy

In [ ]:
h = history or json.loads((OUT / "history.json").read_text())
ep = [r["epoch"] for r in h]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(ep, [r["train_loss"] for r in h], label="train"); a1.plot(ep, [r["val_loss"] for r in h], label="val")
a1.set_title("Loss"); a1.set_xlabel("epoch"); a1.legend()
a2.plot(ep, [r["train_acc"] for r in h], label="train"); a2.plot(ep, [r["val_acc"] for r in h], label="val")
a2.set_title("Accuracy"); a2.set_xlabel("epoch"); a2.legend()
plt.tight_layout(); plt.show()

## 8. ประเมินผลบน Validation set (ใช้โมเดลที่ดีที่สุด)

In [ ]:
model.load_state_dict(torch.load(OUT / "best.pt", map_location=device)["model"])
_, val_pred = evaluate(va)
yv = y_all[va]
val_acc = float((val_pred == yv).mean())
by_class = {classes[c]: float((val_pred[yv == c] == c).mean()) for c in range(len(classes))}
(OUT / "val_report.json").write_text(json.dumps(
    {"source": CONFIG["source"], "best_val_acc": best_acc, "val_acc": val_acc,
     "val_acc_by_class": by_class}, indent=2))

print(f"Validation accuracy: {val_acc:.4f}")
print("\n10 คลาสที่แม่นยำน้อยที่สุด:")
for c, acc in sorted(by_class.items(), key=lambda kv: kv[1])[:10]:
    print(f"  {c} ({thai(c)}): {acc:.4f}")

### Confusion matrix และคู่ที่สับสนบ่อยที่สุด

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(yv, val_pred, labels=range(len(classes)))
plt.figure(figsize=(12, 10))
plt.imshow(np.log1p(cm), cmap="Blues")
plt.xticks(range(len(classes)), classes, rotation=90, fontsize=6)
plt.yticks(range(len(classes)), classes, fontsize=6)
plt.xlabel("predicted"); plt.ylabel("true"); plt.title("Confusion matrix (log scale)")
plt.colorbar(); plt.tight_layout(); plt.show()

off = cm.copy(); np.fill_diagonal(off, 0)
print("คู่ที่ทายสลับกันบ่อยที่สุด (true -> predicted : count)")
for flat in np.argsort(off, axis=None)[::-1][:10]:
    t, p = divmod(flat, len(classes))
    print(f"  {thai(classes[t])} ({classes[t]}) -> {thai(classes[p])} ({classes[p]}) : {off[t, p]}")

### ตัวอย่างรูปที่ทายผิด

In [ ]:
wrong = np.where(val_pred != yv)[0]
pick = np.random.default_rng(1).choice(wrong, min(20, len(wrong)), replace=False)
fig, axes = plt.subplots(2, 10, figsize=(14, 3.5))
for ax, k in zip(axes.flat, pick):
    ax.imshow(x_all[va[k]], cmap="gray"); ax.axis("off")
    ax.set_title(f"{classes[yv[k]]}→{classes[val_pred[k]]}", fontsize=8, color="red")
plt.suptitle("true → predicted (code)"); plt.tight_layout(); plt.show()

## 9. ทำนายรูปใหม่

โมเดลนี้เห็นแต่รูปแบบของ D2 เท่านั้น ถ้าเอาไปทายรูปจากแหล่งอื่นความแม่นยำจะต่ำกว่าที่วัดได้ข้างบน
ลองทายรูปจากแหล่งอื่นดูได้ เพื่อเทียบว่าโมเดลข้ามแหล่งได้ดีแค่ไหน

In [ ]:
@torch.no_grad()
def predict(path, topk=3):
    model.eval()
    img = torch.from_numpy(preprocess(path, CONFIG["img_size"]).copy())[None]
    prob = model(to_input(img, train=False)).softmax(1)[0].cpu()
    top = prob.topk(topk)
    return [(classes[i], thai(classes[i]), float(p)) for p, i in zip(top.values, top.indices)]


samples = [next((ROOT / "Dataset/character/161" / s).iterdir()) for s in ["D1", "D2", "D3"]]
for p in samples:
    print(f"{p.parent.name:3} {p.name}: {predict(p)}")